# Week 4 — Accelerated Gradient Methods
**80-minute computer seminar.** Implement every optimizer from scratch with NumPy. Use library solvers only as references.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True})

## 1. Rotated elongated quadratic
Create $Q=R^T\operatorname{diag}(1,100)R$. The rotation prevents the eigenvectors from coinciding with the plot axes.

In [ ]:
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
Q = R.T @ np.diag([1.0, 100.0]) @ R
f = lambda x: 0.5 * x @ Q @ x
grad = lambda x: Q @ x
eigs = np.linalg.eigvalsh(Q)
mu, L = eigs[0], eigs[-1]
print(f'mu={mu:.3g}, L={L:.3g}, kappa={L/mu:.3g}')

### GD and Polyak momentum
Both methods return the full path and objective history. Add finite-value checks and a maximum iteration guard.

In [ ]:
def gradient_descent(grad, f, x0, alpha, n_iter=150):
    x = np.asarray(x0, float).copy()
    path, values = [x.copy()], [float(f(x))]
    for _ in range(n_iter):
        x = x - alpha * grad(x)
        path.append(x.copy()); values.append(float(f(x)))
    return {'x': np.array(path), 'f': np.array(values)}

def heavy_ball(grad, f, x0, alpha, beta, n_iter=150):
    x = np.asarray(x0, float).copy()
    v = np.zeros_like(x)
    path, values = [x.copy()], [float(f(x))]
    for _ in range(n_iter):
        # TODO: update velocity, then position. Stop if values become non-finite.
        raise NotImplementedError
    return {'x': np.array(path), 'f': np.array(values)}

In [ ]:
alpha_gd = 2 / (L + mu)
alpha_hb = 4 / (np.sqrt(L) + np.sqrt(mu))**2
beta_hb = ((np.sqrt(L) - np.sqrt(mu)) / (np.sqrt(L) + np.sqrt(mu)))**2
x0 = np.array([3.0, 2.0])
print(alpha_gd, alpha_hb, beta_hb)
# TODO: run GD and heavy-ball and compare objective error against iteration count.


### Trajectory visualization
Overlay every method on the same contour levels. Limit each displayed path to a useful number of early iterates.

In [ ]:
def plot_paths(Q, runs, xlim=(-3.5, 3.5), ylim=(-2.5, 2.5)):
    xs = np.linspace(*xlim, 350); ys = np.linspace(*ylim, 350)
    X1, X2 = np.meshgrid(xs, ys)
    Z = 0.5 * (Q[0,0]*X1**2 + 2*Q[0,1]*X1*X2 + Q[1,1]*X2**2)
    fig, ax = plt.subplots()
    ax.contour(X1, X2, Z, levels=np.geomspace(0.05, 500, 18), cmap='Blues')
    for name, run in runs.items():
        path = run['x']
        ax.plot(path[:,0], path[:,1], 'o-', ms=2.5, lw=1, label=name)
    ax.set(xlabel='$x_1$', ylabel='$x_2$', aspect='equal'); ax.legend()
    return fig, ax
# TODO: call plot_paths with a sensible slice of each trajectory.


## 2. Stability region for heavy-ball momentum
Classify each $(\alpha,\beta)$ pair empirically. Then compare with the analytic root condition for every eigenvalue.

In [ ]:
alphas = np.linspace(0.001, 0.06, 100)
betas = np.linspace(0.0, 0.99, 100)

def spectral_radius_hb(alpha, beta, eigenvalues):
    radius = 0.0
    for lam in eigenvalues:
        roots = np.roots([1.0, -(1 + beta - alpha*lam), beta])
        radius = max(radius, np.max(np.abs(roots)))
    return radius

# TODO: fill empirical_state and analytic_stable arrays over the grid.
# Suggested empirical states: 0 divergent, 1 stable-slow, 2 stable-fast.
# Plot empirical states and overlay the analytic rho=1 contour.


## 3. Coordinate descent
For a quadratic, $L_i=Q_{ii}$ and the exact coordinate step is $-\nabla_i f(x)/Q_{ii}$. Compare iterations and equivalent passes.

In [ ]:
def coordinate_descent(Q, b, x0, selector='cyclic', n_updates=1000, seed=0):
    Q, b = np.asarray(Q, float), np.asarray(b, float)
    x = np.asarray(x0, float).copy()
    rng = np.random.default_rng(seed)
    history = {'f': [], 'grad_norm': [], 'coordinate': []}
    for k in range(n_updates):
        g = Q @ x - b
        if selector == 'cyclic':
            i = k % len(x)
        elif selector == 'random':
            i = rng.integers(len(x))
        elif selector == 'greedy':
            # TODO: Gauss-Southwell-Lipschitz choice
            raise NotImplementedError
        else:
            raise ValueError('unknown selector')
        # TODO: apply the exact coordinate update and record history
        raise NotImplementedError
    return x, history

# TODO: construct a 20-dimensional SPD quadratic and compare all selectors.


## 4. Conjugate gradients
Implement CG using one matrix-vector product per iteration. Stop when $\|r_k\|/\|b\|\leq\text{tol}$.

In [ ]:
def conjugate_gradient(A, b, x0=None, tol=1e-10, max_iter=None):
    A, b = np.asarray(A, float), np.asarray(b, float)
    n = len(b); x = np.zeros(n) if x0 is None else np.asarray(x0, float).copy()
    max_iter = n if max_iter is None else max_iter
    r = b - A @ x
    p = r.copy()
    rr = r @ r
    history = {'relative_residual': [np.sqrt(rr)/np.linalg.norm(b)], 'x': [x.copy()]}
    for _ in range(max_iter):
        # TODO: compute Ap, alpha, x_new, r_new, beta, and p_new
        # Record one matrix-vector product and guard against non-positive p.T@A@p.
        raise NotImplementedError
    return x, history


### CG diagnostics
Store residuals and directions temporarily. Inspect $R^TR$ and $P^TAP$ after normalizing their diagonals. In exact arithmetic both should be diagonal.

In [ ]:
rng = np.random.default_rng(4)
n = 40
U, _ = np.linalg.qr(rng.normal(size=(n, n)))
spectrum = np.geomspace(1.0, 1_000.0, n)
A = U @ np.diag(spectrum) @ U.T
b = rng.normal(size=n)
x_direct = np.linalg.solve(A, b)
# TODO: solve with CG and with GD using alpha=2/(L+mu).
# Compare error, residual, time, and matrix-vector products in a table.


## 5. SGD from scratch
Minimize mean squared error. Shuffle each epoch, include the final short batch, and log after complete epochs.

In [ ]:
def make_regression(n=2000, p=12, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    w = rng.normal(size=p)
    y = X @ w + 0.2*rng.normal(size=n)
    return X, y, w

def mse_loss(X, y, w):
    r = X @ w - y
    return 0.5 * np.mean(r*r)

def sgd_mse(X, y, w0, lr_schedule, batch_size=16, epochs=30, seed=0):
    rng = np.random.default_rng(seed)
    w = np.asarray(w0, float).copy()
    n, updates = len(y), 0
    history = {'loss': [mse_loss(X, y, w)], 'examples': [0], 'w': [w.copy()]}
    for epoch in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start+batch_size]
            # TODO: calculate the batch gradient and update with lr_schedule(updates)
            updates += 1
        history['loss'].append(mse_loss(X, y, w))
        history['examples'].append((epoch+1)*n)
        history['w'].append(w.copy())
    return w, history


### Learning-rate schedules and stochastic variability
Compare constant, $1/\sqrt{k}$, and $1/k$ schedules. Repeat with at least five seeds and show median plus interquartile band.

In [ ]:
constant = lambda k: 0.02
inverse_sqrt = lambda k: 0.15 / np.sqrt(k + 1)
inverse = lambda k: 0.5 / (k + 10)
# TODO: compare schedules and batch sizes 1, 16, and full batch by processed examples.


## 6. Feature standardization
Rescale columns, compare condition numbers, and rerun SGD with identical hyperparameters. Convert standardized coefficients back before comparing them.

In [ ]:
X, y, w_true = make_regression()
scales = np.geomspace(1.0, 1_000.0, X.shape[1])
X_bad = X * scales

def standardize(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    if np.any(scale == 0):
        raise ValueError('constant feature')
    return (X-mean)/scale, mean, scale

X_std, mean, scale = standardize(X_bad)
cond_bad = np.linalg.cond(X_bad.T @ X_bad / len(X_bad))
cond_std = np.linalg.cond(X_std.T @ X_std / len(X_std))
print(f'condition numbers: raw={cond_bad:.3e}, standardized={cond_std:.3e}')
# TODO: find stable learning-rate ranges and compare training curves.
# For a model with an intercept, transform coefficients and intercept back to original units.


## 7. Results table and reflection
Summarize each method with its final error and a fair work measure. Explain momentum instability, the gain from conjugacy, the SGD noise floor, and the part of poor conditioning that standardization can remove.

In [ ]:
# TODO: create the final comparison table from recorded histories.
# Optional display helper: import pandas as pd; pd.DataFrame(rows)
